# Week 01 · Day 05 — Local LLMs

**Topics:** Ollama API calls · Quantization · llama.cpp · Local vs API decision

> **Prerequisites:** Install Ollama from https://ollama.ai and run `ollama pull llama3.2` before this notebook.

In [ ]:
%pip install -q openai requests

## 1 · Ollama — OpenAI-Compatible API

Ollama runs local models and exposes them at `http://localhost:11434`. The OpenAI-compatible endpoint means you can use the exact same `openai` library you already know.

In [ ]:
from openai import OpenAI

# Point to local Ollama — no API key needed
ollama = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",  # required by the client but not validated
)

response = ollama.chat.completions.create(
    model="llama3.2",  # or any model you've pulled: llama3.1, mistral, phi3, etc.
    messages=[
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user", "content": "What is the difference between a list and a tuple in Python?"},
    ],
    temperature=0.3,
)

print(response.choices[0].message.content)

In [ ]:
# Streaming with local model
import time

t0 = time.perf_counter()
stream = ollama.chat.completions.create(
    model="llama3.2",
    messages=[{"role": "user", "content": "Explain gradient descent in 3 sentences."}],
    stream=True,
)

tokens = 0
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)
        tokens += 1

elapsed = time.perf_counter() - t0
print(f"\n\nGenerated ~{tokens} tokens in {elapsed:.1f}s ({tokens/elapsed:.1f} tok/s)")

## 2 · Ollama REST API (No Library)

In [ ]:
import requests, json

# List available models
try:
    r = requests.get("http://localhost:11434/api/tags", timeout=5)
    models = r.json().get("models", [])
    print("Installed models:")
    for m in models:
        size_gb = m.get("size", 0) / 1e9
        print(f"  {m['name']:<30} {size_gb:.1f} GB")
except requests.exceptions.ConnectionError:
    print("Ollama not running — start it with: ollama serve")

In [ ]:
# Raw generate API (faster, no chat template overhead)
try:
    r = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "llama3.2",
            "prompt": "List 3 Python best practices.",
            "stream": False,
            "options": {"temperature": 0.3, "num_predict": 150},
        },
        timeout=60,
    )
    data = r.json()
    print(data["response"])
    print(f"\nTokens: {data['eval_count']} in {data['eval_duration']/1e9:.1f}s")
except requests.exceptions.ConnectionError:
    print("Ollama not running")

## 3 · Quantization

Quantization reduces model precision to lower memory requirements. The GGUF format is used by llama.cpp and Ollama.

| Quantization | Bits | Size (7B) | Quality loss | Use case |
|-------------|------|-----------|-------------|----------|
| F16 | 16 | ~13 GB | None | GPU with 16GB+ VRAM |
| Q8_0 | 8 | ~7 GB | Minimal | Best quality/size balance |
| Q4_K_M | 4 | ~4 GB | Small | **Default recommendation** |
| Q3_K_M | 3 | ~3 GB | Moderate | RAM-constrained |
| Q2_K | 2 | ~2.5 GB | Large | Last resort |

In [ ]:
# Compare quantization levels (run if you have multiple quantizations installed)
# ollama pull llama3.2          # default Q4_K_M
# ollama pull llama3.2:8b-q8_0  # Q8_0

test_prompt = "What is 7 × 8 × 9? Show your work."

quantizations = [
    ("llama3.2", "Q4_K_M (default)"),
    # ("llama3.2:8b-q8_0", "Q8_0"),  # uncomment if pulled
]

for model, label in quantizations:
    try:
        t0 = time.perf_counter()
        r = ollama.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": test_prompt}],
            temperature=0,
            max_tokens=100,
        )
        elapsed = time.perf_counter() - t0
        answer = r.choices[0].message.content
        print(f"{label} ({elapsed:.1f}s): {answer[:100]}")
    except Exception as e:
        print(f"{label}: not available ({e})")

## 4 · Local vs API Decision Framework

The choice between running locally and using an API is a system design decision, not a technical limitation.

In [ ]:
def decide_local_or_api(scenario: dict) -> str:
    reasons_local = []
    reasons_api = []

    if scenario.get("data_sensitivity") == "high":
        reasons_local.append("sensitive data — can't send to external API")
    if scenario.get("monthly_requests", 0) > 1_000_000:
        reasons_local.append("high volume — local is cheaper at scale")
    if scenario.get("latency_requirement_ms", 1000) < 100:
        reasons_local.append("ultra-low latency — no network hop")
    if scenario.get("gpu_available") is False:
        reasons_api.append("no GPU — local inference will be too slow")
    if scenario.get("needs_latest_model") is True:
        reasons_api.append("need latest capability — frontier models only via API")
    if scenario.get("team_size", 1) == 1 and scenario.get("prototype") is True:
        reasons_api.append("prototyping solo — API is fastest to start")

    if len(reasons_local) > len(reasons_api):
        return f"LOCAL — {', '.join(reasons_local)}"
    else:
        return f"API — {', '.join(reasons_api) or 'default choice'}"

scenarios = [
    {"data_sensitivity": "high", "monthly_requests": 500_000, "gpu_available": True},
    {"needs_latest_model": True, "gpu_available": False, "prototype": True, "team_size": 1},
    {"monthly_requests": 2_000_000, "gpu_available": True, "data_sensitivity": "low"},
]

for s in scenarios:
    print(f"Scenario: {s}")
    print(f"Decision: {decide_local_or_api(s)}\n")

## 5 · Benchmarking Local vs Cloud Latency

In [ ]:
import os
from openai import OpenAI

oai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
prompt = "Name 5 Python standard library modules."

# Cloud (OpenAI)
t0 = time.perf_counter()
r = oai.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    temperature=0,
    max_tokens=100,
)
cloud_time = time.perf_counter() - t0

# Local (Ollama)
try:
    t0 = time.perf_counter()
    r2 = ollama.chat.completions.create(
        model="llama3.2",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=100,
    )
    local_time = time.perf_counter() - t0
    print(f"Cloud (gpt-4o-mini): {cloud_time:.2f}s")
    print(f"Local (llama3.2):    {local_time:.2f}s")
except Exception:
    print(f"Cloud (gpt-4o-mini): {cloud_time:.2f}s")
    print("Local: Ollama not running")

## Summary

- Ollama exposes a local server at `:11434` with an OpenAI-compatible `/v1` endpoint.
- Use the `openai` Python library with `base_url="http://localhost:11434/v1"` — same code, local model.
- Q4_K_M is the recommended default quantization: good quality, runs on 8GB RAM.
- Choose local when: data is sensitive, volume is very high, or latency must be sub-100ms.
- Choose API when: prototyping, no GPU, or frontier model quality is required.

**Week 01 complete!** Next: [Week 02 — LangChain](week02-day01-langchain.ipynb)